# 🧠 Brain Tumor Segmentation — 3D SegResNet + MONAI

[![Python](https://img.shields.io/badge/Python-3.8%2B-3776AB?logo=python&logoColor=white)](https://www.python.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.0%2B-EE4C2C?logo=pytorch&logoColor=white)](https://pytorch.org/)
[![MONAI](https://img.shields.io/badge/MONAI-1.3%2B-0096FF)](https://monai.io/)
[![Dataset](https://img.shields.io/badge/Dataset-BraTS%202021-20BEFF?logo=kaggle)](https://www.kaggle.com/datasets/dschettler8845/brats-2021-task1)
[![License](https://img.shields.io/badge/License-MIT-yellow.svg)](../LICENSE)

> **Volumetric 3D brain tumor segmentation** on BraTS 2021 using **SegResNet** (MONAI).  
> Predicts three clinically relevant tumor sub-regions simultaneously:  
> **TC** (Tumor Core) · **WT** (Whole Tumor) · **ET** (Enhancing Tumor)

---

## 📋 Notebook Structure

| Cell | Purpose | Run order |
|------|---------|-----------|
| **Cell 1** | Install · Imports · Data · Model · Optimiser — full setup | 1st — always |
| **Cell 2** | Training loop (fresh start **or** auto-resume from checkpoint) | 2nd |
| **Cell 3** | Training curves & metric visualisation | After Cell 2 |
| **Cell 4** | GT vs Prediction overlay + per-patient Dice table | After Cell 2 |

> ⚠️ **GPU required.** Enable *Settings → Accelerator → GPU T4* on Kaggle.  
> ⚠️ **Disk limit.** Kaggle caps working space at ~20 GB. The pipeline extracts only what it needs.

---

### Expected Results (150 epochs, Kaggle T4)

| Metric | TC | WT | ET | Mean |
|--------|----|----|-----|------|
| **Val Dice** | ~0.78 | ~0.88 | ~0.72 | **~0.79** |

> Training time: ~8–15 min/epoch on T4 · ~20–38 h total with resume sessions

## ⚙️ Cell 1 — Setup

Installs MONAI, imports all libraries, extracts data, builds datasets, model, and optimiser.  
**Must run before any other cell.**

In [ ]:
# ============================================================================
# CELL 1 — SETUP
# BraTS 2021 · 3D SegResNet · Full pipeline initialisation
# ============================================================================

# ── 1.1  Install MONAI ────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "monai[all]", "-q"],
               check=True)

# ── 1.2  Imports ──────────────────────────────────────────────────────────────
import os, random, glob, time, copy, tarfile, sys, warnings
from datetime import timedelta

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast

import monai
from monai.data import Dataset, DataLoader, decollate_batch
from monai.losses import DiceLoss, FocalLoss
from monai.metrics import DiceMetric
from monai.networks.nets import SegResNet
from monai.transforms import (
    Activations, AsDiscrete, Compose,
    LoadImaged, EnsureChannelFirstd,
    ConvertToMultiChannelBasedOnBratsClassesd,
    Orientationd, Spacingd, NormalizeIntensityd,
    CropForegroundd, SpatialPadd, RandSpatialCropd,
    RandFlipd, RandRotate90d, RandZoomd,
    RandGaussianNoised, RandScaleIntensityd, RandShiftIntensityd,
    RandAdjustContrastd, ToTensord,
)
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

# ── 1.3  Reproducibility & device ─────────────────────────────────────────────
SEED = 42
set_determinism(seed=SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device    : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"   GPU       : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   PyTorch   : {torch.__version__}")
print(f"   MONAI     : {monai.__version__}")

# ─────────────────────────────────────────────────────────────────────────────
# §0  LOCATE & EXTRACT DATASET
# ─────────────────────────────────────────────────────────────────────────────
BASE_INPUT    = "/kaggle/input"
WORKING_DIR   = "/kaggle/working"
EXTRACTED_DIR = os.path.join(WORKING_DIR, "BraTS2021_Extracted")

os.makedirs(EXTRACTED_DIR, exist_ok=True)

if not os.listdir(EXTRACTED_DIR):
    tar_files = []
    for root, _, files in os.walk(BASE_INPUT):
        for f in files:
            if f.endswith((".tar", ".tar.gz")):
                tar_files.append(os.path.join(root, f))

    if not tar_files:
        raise FileNotFoundError(
            f"No .tar / .tar.gz found under {BASE_INPUT}. "
            "Please attach the BraTS 2021 dataset to this notebook."
        )

    for tp in tar_files:
        print(f"📦  Extracting {os.path.basename(tp)} …")
        with tarfile.open(tp, "r") as tar:
            tar.extractall(path=EXTRACTED_DIR)
        print(f"    Done.")

case_dirs = []
for root, _, files in os.walk(EXTRACTED_DIR):
    if len([f for f in files if f.endswith(".nii.gz")]) >= 5:
        case_dirs.append(root)
case_dirs = sorted(case_dirs)

if not case_dirs:
    raise RuntimeError(
        "No valid case directories found after extraction. "
        "Expected folders with 5+ .nii.gz files each."
    )
print(f"✅  Found {len(case_dirs)} cases")

# ─────────────────────────────────────────────────────────────────────────────
# §1  HYPER-PARAMETERS  (all tuneable from one dict)
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # Data
    "patch_size"   : (128, 128, 128),
    "num_workers"  : 2,
    # Training
    "max_epochs"   : 150,
    "batch_size"   : 1,           # 3D volumes are large — keep at 1
    "grad_accum"   : 2,           # effective batch = 1 × 2 = 2
    "clip_grad"    : 1.0,
    "patience"     : 20,          # early-stopping patience
    "ckpt_every"   : 3,           # save checkpoint every N epochs
    # Optimiser
    "lr"           : 2e-4,
    "weight_decay" : 1e-4,
    "betas"        : (0.9, 0.999),
    # Scheduler
    "warmup_epochs": 10,
    "eta_min"      : 1e-6,
    # EMA
    "ema_decay"    : 0.9998,
    # Sliding-window inference
    "sw_roi_size"  : (128, 128, 128),
    "sw_overlap"   : 0.25,
    "sw_batch_size": 2,
    # Model
    "blocks_down"  : [1, 2, 2, 4],
    "blocks_up"    : [1, 1, 1],
    "init_filters" : 32,
    "dropout"      : 0.1,
}

# ─────────────────────────────────────────────────────────────────────────────
# §2  BUILD FILE LIST & TRAIN / VAL / TEST SPLIT  (80 / 10 / 10)
# ─────────────────────────────────────────────────────────────────────────────
def build_file_list(dirs):
    items = []
    for cd in dirs:
        nifti_files = glob.glob(os.path.join(cd, "*.nii.gz"))
        flair = t1ce = t1 = t2 = seg = None
        for nf in nifti_files:
            b = os.path.basename(nf).lower()
            if   "flair" in b:                      flair = nf
            elif "t1ce"  in b:                      t1ce  = nf
            elif "t1."   in b and "t1ce" not in b:  t1    = nf
            elif "t2."   in b:                      t2    = nf
            elif "seg"   in b:                      seg   = nf
        if all([flair, t1ce, t1, t2, seg]):
            items.append({"image": [flair, t1ce, t1, t2], "label": seg})
    return items

all_files = build_file_list(case_dirs)
random.seed(SEED)
random.shuffle(all_files)

n       = len(all_files)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_files = all_files[:n_train]
val_files   = all_files[n_train : n_train + n_val]
test_files  = all_files[n_train + n_val :]

print(f"📊  Split — Train : {len(train_files)} | Val : {len(val_files)} | Test : {len(test_files)}")

# ─────────────────────────────────────────────────────────────────────────────
# §3  MONAI TRANSFORMS
# ─────────────────────────────────────────────────────────────────────────────
P = CFG["patch_size"]

# Shared preprocessing applied to train, val, and test
shared_pre = [
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),
    # BraTS labels 0=BG, 1=NCR, 2=ED, 4=ET → 3-channel binary (TC, WT, ET)
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    Orientationd(keys=["image", "label"], axcodes="RAS", labels=None),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0),
             mode=("bilinear", "nearest")),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CropForegroundd(keys=["image", "label"], source_key="image",
                    select_fn=lambda x: x > 0, margin=10),
    SpatialPadd(keys=["image", "label"], spatial_size=P),
]

# Training augmentation
train_aug = [
    RandSpatialCropd(keys=["image", "label"], roi_size=P, random_size=False),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    RandZoomd(keys=["image", "label"], min_zoom=0.9, max_zoom=1.1,
              mode=("trilinear", "nearest"), prob=0.3),
    RandGaussianNoised(keys="image", std=0.05, prob=0.25),
    RandScaleIntensityd(keys="image", factors=0.1, prob=0.3),
    RandShiftIntensityd(keys="image", offsets=0.1, prob=0.3),
    RandAdjustContrastd(keys="image", gamma=(0.7, 1.5), prob=0.3),
    ToTensord(keys=["image", "label"]),
]

val_pre = [
    SpatialPadd(keys=["image", "label"], spatial_size=P),
    ToTensord(keys=["image", "label"]),
]

train_transforms = Compose(shared_pre + train_aug)
val_transforms   = Compose(shared_pre + val_pre)
test_transforms  = Compose(shared_pre + val_pre)

# ─────────────────────────────────────────────────────────────────────────────
# §4  DATASETS & DATA LOADERS
# ─────────────────────────────────────────────────────────────────────────────
train_ds = Dataset(train_files, transform=train_transforms)
val_ds   = Dataset(val_files,   transform=val_transforms)
test_ds  = Dataset(test_files,  transform=test_transforms)

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=1, shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=1, shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True,
)

print(f"🔄  Loaders — Train batches : {len(train_loader)} | Val : {len(val_loader)} | Test : {len(test_loader)}")

# ─────────────────────────────────────────────────────────────────────────────
# §5  MODEL — SegResNet + EMA shadow copy
# ─────────────────────────────────────────────────────────────────────────────
model = SegResNet(
    blocks_down  = CFG["blocks_down"],
    blocks_up    = CFG["blocks_up"],
    init_filters = CFG["init_filters"],
    in_channels  = 4,   # FLAIR, T1ce, T1, T2
    out_channels = 3,   # TC, WT, ET
    dropout_prob = CFG["dropout"],
).to(DEVICE)

# EMA shadow model — used for validation (smoother, better generalisation)
ema_model = copy.deepcopy(model)
ema_model.eval()

@torch.no_grad()
def update_ema(ema, live, decay):
    """Exponential moving average weight update."""
    for ep, lp in zip(ema.parameters(), live.parameters()):
        ep.data.mul_(decay).add_(lp.data, alpha=1.0 - decay)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🧠  Model     : SegResNet — {n_params:,} trainable parameters")

# ─────────────────────────────────────────────────────────────────────────────
# §6  LOSS — Dice + Focal
# ─────────────────────────────────────────────────────────────────────────────
dice_loss_fn  = DiceLoss(sigmoid=True, batch=True, smooth_nr=1e-5, smooth_dr=1e-5)
focal_loss_fn = FocalLoss(gamma=2.0)

def combined_loss(pred, target):
    """Dice + Focal — Dice maximises overlap, Focal penalises hard boundary voxels."""
    return dice_loss_fn(pred, target) + focal_loss_fn(pred, target)

# ─────────────────────────────────────────────────────────────────────────────
# §7  OPTIMISER & SCHEDULERS
# ─────────────────────────────────────────────────────────────────────────────
# Separate parameter groups: no weight-decay on biases and normalisation layers
no_decay_params = [p for name, p in model.named_parameters()
                   if any(nd in name for nd in ["bias", "norm"]) and p.requires_grad]
decay_params    = [p for name, p in model.named_parameters()
                   if not any(nd in name for nd in ["bias", "norm"]) and p.requires_grad]

optimizer = optim.AdamW(
    [{"params": decay_params,    "weight_decay": CFG["weight_decay"]},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=CFG["lr"], betas=CFG["betas"],
)

cosine_sched = CosineAnnealingLR(
    optimizer,
    T_max   = CFG["max_epochs"] - CFG["warmup_epochs"],
    eta_min = CFG["eta_min"],
)

warmup_sched = optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0,
    total_iters=CFG["warmup_epochs"],
)

scaler = GradScaler()

# ─────────────────────────────────────────────────────────────────────────────
# §8  METRICS & POST-PROCESSING
# ─────────────────────────────────────────────────────────────────────────────
dice_metric = DiceMetric(include_background=True, reduction="mean_batch")
post_pred   = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])

# ─────────────────────────────────────────────────────────────────────────────
# §9  DISPLAY HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def fmt_time(sec):
    return str(timedelta(seconds=int(sec)))

def _bar(pct, width=28):
    filled = int(width * pct)
    return "█" * filled + "░" * (width - filled)

def print_epoch_header():
    h = (f"{'Ep':>4} │ {'Loss':>8} │ {'mDice':>7} │ "
         f"{'TC':>7} │ {'WT':>7} │ {'ET':>7} │ "
         f"{'Ep.Time':>8} │ {'Elapsed':>8} │ {'ETA':>8} │ Flag")
    print("─" * len(h))
    print(h)
    print("─" * len(h))

def print_epoch_row(ep, loss, dice, tc, wt, et, ep_s, ela_s, eta_s, best):
    flag = " ★ BEST" if best else ""
    print(f"{ep:>4} │ {loss:>8.4f} │ {dice:>7.4f} │ "
          f"{tc:>7.4f} │ {wt:>7.4f} │ {et:>7.4f} │ "
          f"{fmt_time(ep_s):>8} │ {fmt_time(ela_s):>8} │ {fmt_time(eta_s):>8} │{flag}",
          flush=True)

def progress(ep, total_ep, step, total_steps, loss, t0):
    pct = step / total_steps
    ela = time.time() - t0
    eta = (ela / pct - ela) if pct > 0 else 0
    sys.stdout.write(
        f"\r  Epoch {ep}/{total_ep}  [{_bar(pct)}] {step}/{total_steps}"
        f"  loss={loss:.4f}  ela={fmt_time(ela)}  eta={fmt_time(eta)}  "
    )
    sys.stdout.flush()

# ─────────────────────────────────────────────────────────────────────────────
# §10  CHECKPOINT HELPERS
# ─────────────────────────────────────────────────────────────────────────────
_last_ckpt_path = None

def save_checkpoint(epoch, best_dice, history):
    """Save full training state; delete previous checkpoint to save disk space."""
    global _last_ckpt_path
    path = os.path.join(WORKING_DIR, f"checkpoint_epoch_{epoch:03d}.pth")
    torch.save({
        "epoch":                epoch,
        "model_state_dict":     model.state_dict(),
        "ema_state_dict":       ema_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": cosine_sched.state_dict(),
        "scaler_state_dict":    scaler.state_dict(),
        "best_dice":            best_dice,
        "history":              history,
        "cfg":                  CFG,
    }, path)
    if _last_ckpt_path and os.path.exists(_last_ckpt_path):
        os.remove(_last_ckpt_path)
        print(f"  🗑️   Deleted old checkpoint: {os.path.basename(_last_ckpt_path)}",
              flush=True)
    _last_ckpt_path = path
    print(f"  💾  Checkpoint → {os.path.basename(path)}", flush=True)

def load_checkpoint(path):
    """Restore full training state from .pth file."""
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    ema_model.load_state_dict(ckpt["ema_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    cosine_sched.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    return ckpt["epoch"] + 1, ckpt["best_dice"], ckpt["history"]

# ─────────────────────────────────────────────────────────────────────────────
# §11  SANITY CHECK
# ─────────────────────────────────────────────────────────────────────────────
required = {
    "model": model, "ema_model": ema_model,
    "optimizer": optimizer, "cosine_sched": cosine_sched, "scaler": scaler,
    "train_loader": train_loader, "val_loader": val_loader, "test_loader": test_loader,
    "dice_metric": dice_metric, "post_pred": post_pred,
    "DEVICE": DEVICE, "CFG": CFG, "WORKING_DIR": WORKING_DIR,
}
all_ok = True
for name, obj in required.items():
    ok = obj is not None
    if not ok:
        all_ok = False
    print(f"  {'✅' if ok else '❌'} {name}")

print()
if all_ok:
    print("✅  Cell 1 complete — all objects ready. Proceed to Cell 2.")
else:
    print("❌  Some objects missing — fix errors above before continuing.")

## 🏋️ Cell 2 — Training

Runs the full training loop — **fresh start or automatic resume** from a checkpoint.

**Checkpoint resume:** Upload a `checkpoint_epoch_XXX.pth` as a Kaggle Dataset and attach it.  
Cell 2 auto-detects any `.pth` in `/kaggle/input/` and resumes from that epoch.

> Run Cell 1 first.

In [ ]:
# ============================================================================
# CELL 2 — TRAINING LOOP
# Fresh start OR automatic resume from checkpoint
# Requires Cell 1 to be executed first
# ============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# §A  AUTO-DETECT CHECKPOINT
# ─────────────────────────────────────────────────────────────────────────────
_pth_files = sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True))
_resume    = _pth_files[-1] if _pth_files else None

if _resume:
    print(f"📂  Checkpoint found : {_resume}")
else:
    print("🆕  No checkpoint found — starting from scratch.")

# ─────────────────────────────────────────────────────────────────────────────
# §B  INITIALISE TRAINING STATE
# ─────────────────────────────────────────────────────────────────────────────
if _resume:
    start_epoch, best_dice, history = load_checkpoint(_resume)
    no_improve = 0
    print(f"✅  Resumed from epoch {start_epoch - 1} | best Dice = {best_dice:.4f}")
else:
    start_epoch = 1
    best_dice   = -1.0
    no_improve  = 0
    history     = {
        "loss":     [],
        "val_dice": [],
        "val_tc":   [],
        "val_wt":   [],
        "val_et":   [],
        "lr":       [],
    }

end_epoch   = CFG["max_epochs"]
train_start = time.time()

# ─────────────────────────────────────────────────────────────────────────────
# §C  TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
print_epoch_header()

for epoch in range(start_epoch, end_epoch + 1):

    # ── Training phase ────────────────────────────────────────────────────────
    model.train()
    epoch_start  = time.time()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader, 1):
        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        with autocast():
            preds = model(images)
            loss  = combined_loss(preds, labels) / CFG["grad_accum"]

        scaler.scale(loss).backward()

        if step % CFG["grad_accum"] == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["clip_grad"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            update_ema(ema_model, model, CFG["ema_decay"])

        running_loss += loss.item() * CFG["grad_accum"]
        progress(epoch, end_epoch, step, len(train_loader), running_loss / step, epoch_start)

    sys.stdout.write("\n")

    # ── LR scheduling ─────────────────────────────────────────────────────────
    cur_lr = optimizer.param_groups[0]["lr"]
    if epoch <= CFG["warmup_epochs"]:
        warmup_sched.step()
    else:
        cosine_sched.step()

    epoch_loss = running_loss / len(train_loader)
    history["loss"].append(epoch_loss)
    history["lr"].append(cur_lr)

    # ── Validation phase (on EMA model) ──────────────────────────────────────
    ema_model.eval()
    dice_metric.reset()

    with torch.no_grad():
        for v_step, batch in enumerate(val_loader, 1):
            images = batch["image"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)

            with autocast():
                preds = sliding_window_inference(
                    inputs        = images,
                    roi_size      = CFG["sw_roi_size"],
                    sw_batch_size = CFG["sw_batch_size"],
                    predictor     = ema_model,
                    overlap       = CFG["sw_overlap"],
                )

            preds_bin   = [post_pred(x) for x in decollate_batch(preds)]
            labels_list = decollate_batch(labels)
            dice_metric(y_pred=preds_bin, y=labels_list)

            sys.stdout.write(
                f"\r  Validation [{_bar(v_step / len(val_loader), 20)}]"
                f" {v_step}/{len(val_loader)}  "
            )
            sys.stdout.flush()

    sys.stdout.write("\n")

    vals       = dice_metric.aggregate()   # shape (3,)
    tc, wt, et = vals[0].item(), vals[1].item(), vals[2].item()
    mean_dice  = (tc + wt + et) / 3.0

    history["val_dice"].append(mean_dice)
    history["val_tc"].append(tc)
    history["val_wt"].append(wt)
    history["val_et"].append(et)

    # ── Timing ────────────────────────────────────────────────────────────────
    ep_sec  = time.time() - epoch_start
    ela_sec = time.time() - train_start
    rem_eps = end_epoch - epoch
    eta_sec = (ela_sec / epoch) * rem_eps if epoch > 0 else 0

    is_best = mean_dice > best_dice
    print_epoch_row(epoch, epoch_loss, mean_dice, tc, wt, et,
                    ep_sec, ela_sec, eta_sec, is_best)

    # ── Save best model ────────────────────────────────────────────────────────
    if is_best:
        best_dice  = mean_dice
        no_improve = 0
        torch.save(ema_model.state_dict(),
                   os.path.join(WORKING_DIR, "best_model.pth"))
        print(f"  🏆  New best → best_model.pth  (mean Dice = {best_dice:.4f})")
    else:
        no_improve += 1

    # ── Periodic checkpoint ────────────────────────────────────────────────────
    if epoch % CFG["ckpt_every"] == 0:
        save_checkpoint(epoch, best_dice, history)

    # ── Early stopping ─────────────────────────────────────────────────────────
    if no_improve >= CFG["patience"]:
        print(f"\n⛔  Early stopping — no improvement for {CFG['patience']} epochs.")
        break

# ─────────────────────────────────────────────────────────────────────────────
# §D  TRAINING COMPLETE SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
best_ep = int(np.argmax(history["val_dice"]))
print("\n" + "═" * 80)
print(f"  Training complete  |  Total time : {fmt_time(time.time() - train_start)}")
print(f"  Best Val Mean Dice : {best_dice:.4f}  (epoch {best_ep + 1})")
print(f"    TC = {history['val_tc'][best_ep]:.4f}  "
      f"WT = {history['val_wt'][best_ep]:.4f}  "
      f"ET = {history['val_et'][best_ep]:.4f}")
print(f"  Outputs → {WORKING_DIR}/")
print("═" * 80)

## 📈 Cell 3 — Training Curves

Plots training loss and per-class Dice metrics over epochs, with smoothed curves, best-point annotations, and a summary panel.  
Run after Cell 2.

In [ ]:
# ============================================================================
# CELL 3 — TRAINING CURVES & METRIC VISUALISATION
# Requires: history dict (populated in Cell 2)
# ============================================================================

from scipy.ndimage import uniform_filter1d
from matplotlib.ticker import MaxNLocator

# ── Style ─────────────────────────────────────────────────────────────────────
BG_DARK   = "#0d1117"
BG_PANEL  = "#161b22"
BG_PANEL2 = "#1c2128"
BORDER    = "#30363d"
TEXT_PRI  = "#e6edf3"
TEXT_SEC  = "#8b949e"

plt.rcParams.update({
    "figure.facecolor" : BG_DARK,
    "axes.facecolor"   : BG_PANEL,
    "axes.edgecolor"   : BORDER,
    "axes.labelcolor"  : TEXT_SEC,
    "xtick.color"      : TEXT_SEC,
    "ytick.color"      : TEXT_SEC,
    "text.color"       : TEXT_PRI,
    "grid.color"       : BORDER,
    "grid.linestyle"   : "--",
    "grid.alpha"       : 0.4,
    "font.family"      : "DejaVu Sans",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
})

COLORS = {
    "loss":     "#ff6b6b",
    "val_dice": "#2dd4bf",
    "val_tc":   "#fb923c",
    "val_wt":   "#60a5fa",
    "val_et":   "#c084fc",
}

def smooth(v, w=5):
    return uniform_filter1d(v, size=w, mode="nearest")

def dice_quality_color(score):
    """Return a colour reflecting clinical quality threshold."""
    if score >= 0.85: return "#2dd4bf"   # excellent
    if score >= 0.70: return "#fb923c"   # acceptable
    return "#ff6b6b"                      # needs improvement

def annotate_best(ax, epochs, values, color, is_dice, total_epochs):
    best_idx = int(np.argmax(values)) if is_dice else int(np.argmin(values))
    best_val = float(values[best_idx])
    best_ep  = best_idx + 1

    ax.axvline(best_ep, color=color, linestyle=":", alpha=0.45, linewidth=1.3, zorder=1)
    ax.scatter([best_ep], [best_val], color="white", s=200, zorder=6,
               marker="*", edgecolors=color, linewidths=1.5)

    badge_color = dice_quality_color(best_val) if is_dice else color
    offset = (30, -14) if best_ep > total_epochs * 0.62 else (20, 18)
    ax.annotate(
        f"  ★  Best : {best_val:.4f}\n  epoch {best_ep}",
        xy=(best_ep, best_val),
        xytext=offset, textcoords="offset points",
        fontsize=11.5, color="white", fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.45", facecolor=badge_color, alpha=0.85, edgecolor="white"),
        arrowprops=dict(arrowstyle="->", color="white", lw=1.2),
    )

epochs_range  = list(range(1, len(history["loss"]) + 1))
total_epochs  = len(epochs_range)

fig, axes = plt.subplots(2, 3, figsize=(20, 11), facecolor=BG_DARK)
fig.suptitle("BraTS 2021 — SegResNet 3D  |  Training Dashboard",
             fontsize=16, fontweight="bold", color=TEXT_PRI, y=1.01)

# (ax, key, title, is_dice, smooth_w)
plots = [
    (axes[0, 0], "loss",     "Training Loss",             False, 5),
    (axes[0, 1], "val_dice", "Validation Mean Dice",      True,  3),
    (axes[1, 0], "val_tc",   "Dice — TC (Tumor Core)",    True,  3),
    (axes[1, 1], "val_wt",   "Dice — WT (Whole Tumor)",   True,  3),
    (axes[1, 2], "val_et",   "Dice — ET (Enhancing)",     True,  3),
]

for ax, key, title, is_dice, w in plots:
    raw = history[key]
    smo = smooth(raw, w)
    ax.plot(epochs_range, raw, color=COLORS[key], alpha=0.30, linewidth=1)
    ax.plot(epochs_range, smo, color=COLORS[key], linewidth=2.5, label="Smoothed")
    annotate_best(ax, epochs_range, raw, COLORS[key], is_dice, total_epochs)
    ax.set_title(title, color=TEXT_PRI, pad=8, fontsize=12)
    ax.set_xlabel("Epoch", color=TEXT_SEC)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(True, axis="y", alpha=0.3)
    if is_dice:
        ax.set_ylim(0, 1.05)
        ax.axhline(0.85, ls=":", color="white", alpha=0.3, lw=0.8, label="0.85 target")

# ── Summary panel ─────────────────────────────────────────────────────────────
axes[0, 2].set_facecolor(BG_PANEL2)
axes[0, 2].axis("off")

best_ep = int(np.argmax(history["val_dice"]))
best_tc = history["val_tc"][best_ep]
best_wt = history["val_wt"][best_ep]
best_et = history["val_et"][best_ep]

summary = (
    f"  Training Summary\n"
    f"  {'─' * 28}\n"
    f"  Epochs trained : {len(history['loss'])}\n"
    f"  Best epoch     : {best_ep + 1}\n"
    f"  Best mean Dice : {max(history['val_dice']):.4f}\n\n"
    f"  Per-class at best epoch\n"
    f"  {'─' * 28}\n"
    f"  TC  (Tumor Core)      : {best_tc:.4f}\n"
    f"  WT  (Whole Tumor)     : {best_wt:.4f}\n"
    f"  ET  (Enhancing)       : {best_et:.4f}\n\n"
    f"  Final LR       : {history['lr'][-1]:.2e}\n"
    f"  Min train loss : {min(history['loss']):.4f}"
)
axes[0, 2].text(0.05, 0.95, summary,
                transform=axes[0, 2].transAxes,
                fontsize=11.5, color=TEXT_PRI,
                verticalalignment="top",
                fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor=BG_PANEL, alpha=0.95))

plt.tight_layout()
out_path = os.path.join(WORKING_DIR, "training_curves.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=BG_DARK)
plt.show()
print(f"✅  training_curves.png saved → {out_path}")

## 🔬 Cell 4 — Ground Truth vs Prediction

Overlays GT and predicted masks on axial FLAIR slices for N test patients.  
Displays per-patient slice Dice scores and saves `gt_vs_pred.png`.

Run after Cell 2. Requires `best_model.pth` in `/kaggle/working/`.

In [ ]:
# ============================================================================
# CELL 4 — GROUND TRUTH vs PREDICTION OVERLAY
# Requires: best_model.pth, test_files, model, ema_model, DEVICE, CFG
# ============================================================================

from matplotlib.patches import Patch

# ── Reload best EMA model ─────────────────────────────────────────────────────
_best_path = os.path.join(WORKING_DIR, "best_model.pth")
if not os.path.exists(_best_path):
    raise FileNotFoundError(f"best_model.pth not found at {_best_path}. Run Cell 2 first.")

ema_model.load_state_dict(torch.load(_best_path, map_location=DEVICE))
ema_model.eval()
print(f"✅  best_model.pth loaded  (best Val Dice = {best_dice:.4f})")

# ── Class colours and labels ──────────────────────────────────────────────────
CLASS_COLORS = {
    0: (1.00, 0.60, 0.00),   # TC — Orange
    1: (0.00, 0.80, 0.80),   # WT — Cyan
    2: (0.70, 0.20, 0.90),   # ET — Violet
}
CLASS_NAMES = ["TC — Tumor Core", "WT — Whole Tumor", "ET — Enhancing"]

def normalise(arr):
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo + 1e-8)

def overlay(flair_2d, mask_3d, alpha=0.55):
    """Blend binary class masks over a greyscale FLAIR slice."""
    base = np.stack([normalise(flair_2d)] * 3, axis=-1)
    for cls, color in CLASS_COLORS.items():
        m = mask_3d[cls]
        for c in range(3):
            base[:, :, c] = np.where(
                m > 0.5,
                alpha * color[c] + (1 - alpha) * base[:, :, c],
                base[:, :, c],
            )
    return np.clip(base, 0, 1)

# ── Visualisation transforms (full volume, deterministic) ────────────────────
vis_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    Orientationd(keys=["image", "label"], axcodes="RAS", labels=None),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0),
             mode=("bilinear", "nearest")),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CropForegroundd(keys=["image", "label"], source_key="image",
                    select_fn=lambda x: x > 0, margin=10),
    SpatialPadd(keys=["image", "label"], spatial_size=CFG["patch_size"]),
    ToTensord(keys=["image", "label"]),
])

N_VIS      = 4
vis_cases  = test_files[:N_VIS] if len(test_files) >= N_VIS else val_files[:N_VIS]
vis_ds     = Dataset(vis_cases, transform=vis_transforms)
vis_loader = DataLoader(vis_ds, batch_size=1, shuffle=False, num_workers=0)

# ── Layout ────────────────────────────────────────────────────────────────────
# Columns: FLAIR | GT Overlay | Pred Overlay | GT per-class | Pred per-class
n_cols = 5
n_rows = len(vis_cases)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows),
                         facecolor="#0d1117")
fig.suptitle("Ground Truth  vs  Prediction — BraTS 2021 (best axial slice)",
             fontsize=14, color="white", fontweight="bold", y=1.01)

# Column header labels
if n_rows > 0:
    for col_idx, title in enumerate(
        ["FLAIR", "GT Overlay", "Pred Overlay", "GT: TC | WT | ET", "Pred: TC | WT | ET"]
    ):
        axes[0, col_idx].set_title(title, color="#8b949e", fontsize=9, pad=3)

with torch.no_grad():
    for row_idx, batch in enumerate(vis_loader):
        if row_idx >= N_VIS:
            break

        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"]   # keep on CPU

        with autocast():
            preds = sliding_window_inference(
                inputs        = images,
                roi_size      = CFG["sw_roi_size"],
                sw_batch_size = CFG["sw_batch_size"],
                predictor     = ema_model,
                overlap       = CFG["sw_overlap"],
            )

        pred_bin  = post_pred(preds[0]).cpu().numpy()   # (3, D, H, W)
        label_vol = labels[0].numpy()                   # (3, D, H, W)
        flair_vol = images[0, 0].cpu().numpy()          # (D, H, W)  — FLAIR channel

        # Pick axial slice with most WT foreground
        best_z = int(np.argmax(label_vol[1].sum(axis=(1, 2))))

        flair_sl = flair_vol[best_z]
        gt_sl    = label_vol[:, best_z, :, :]
        pred_sl  = pred_bin[:, best_z, :, :]

        # Per-slice Dice scores
        slice_dice = [
            2 * (gt_sl[c] * pred_sl[c]).sum() /
            (gt_sl[c].sum() + pred_sl[c].sum() + 1e-5)
            for c in range(3)
        ]

        # Col 0: FLAIR
        axes[row_idx, 0].imshow(normalise(flair_sl), cmap="gray", interpolation="nearest")
        axes[row_idx, 0].set_ylabel(f"Patient {row_idx + 1}", color="white",
                                     fontsize=9, labelpad=4)
        axes[row_idx, 0].set_xlabel(f"z = {best_z}", color="#8b949e", fontsize=8)

        # Col 1: GT overlay
        axes[row_idx, 1].imshow(overlay(flair_sl, gt_sl), interpolation="nearest")

        # Col 2: Pred overlay
        axes[row_idx, 2].imshow(overlay(flair_sl, pred_sl), interpolation="nearest")

        # Col 3: GT per-class strip
        strip_gt = np.concatenate([gt_sl[c] for c in range(3)], axis=1)
        axes[row_idx, 3].imshow(strip_gt, cmap="hot", vmin=0, vmax=1, interpolation="nearest")

        # Col 4: Pred per-class strip + Dice scores
        strip_pr = np.concatenate([pred_sl[c] for c in range(3)], axis=1)
        axes[row_idx, 4].imshow(strip_pr, cmap="hot", vmin=0, vmax=1, interpolation="nearest")
        axes[row_idx, 4].set_xlabel(
            f"TC={slice_dice[0]:.2f}  WT={slice_dice[1]:.2f}  ET={slice_dice[2]:.2f}",
            color="#2dd4bf", fontsize=8,
        )

        for ax in axes[row_idx]:
            ax.tick_params(left=False, bottom=False,
                           labelleft=False, labelbottom=False)
            for spine in ax.spines.values():
                spine.set_edgecolor("#30363d")

# ── Legend ────────────────────────────────────────────────────────────────────
legend_handles = [
    Patch(facecolor=CLASS_COLORS[i], label=CLASS_NAMES[i]) for i in range(3)
]
fig.legend(
    handles=legend_handles, loc="lower center", ncol=3,
    facecolor="#161b22", labelcolor="white", fontsize=10,
    bbox_to_anchor=(0.5, -0.02), edgecolor="#30363d",
)

plt.tight_layout()
out_path = os.path.join(WORKING_DIR, "gt_vs_pred.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print(f"✅  gt_vs_pred.png saved → {out_path}")